# 02 - Workload Classifier Evaluation

This notebook evaluates the tuned LightGBM classifier and compares it with the ARF+ADWIN streaming model.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import confusion_matrix, classification_report

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.data_loader import load_split

LABEL_NAMES = ["DB_OLTP", "VM", "Backup", "AI_Training", "AI_Inference"]

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

tuned_model = joblib.load(PROJECT_ROOT / "models" / "classifier" / "lightgbm_tuned_model.pkl")
scaler = joblib.load(PROJECT_ROOT / "models" / "scaler.pkl")

with open(PROJECT_ROOT / "models" / "classifier" / "lightgbm_tuned_metrics.json", "r", encoding="utf-8") as f:
    tuned_metrics = json.load(f)

print("Train / Val / Test shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Scaler feature count:", getattr(scaler, "n_features_in_", "n/a"))
print("Tuned model type:", type(tuned_model).__name__)

In [ ]:
# Print tuned LightGBM metrics and target status
val_acc = tuned_metrics["val"]["accuracy"]
test_acc = tuned_metrics["test"]["accuracy"]
print(f"Validation accuracy: {val_acc:.4f}")
print(f"Test accuracy:       {test_acc:.4f}")
print("HPE ≥95% target met on test:", "YES" if test_acc >= 0.95 else "NO")

preds = tuned_model.predict(X_test)
cm = confusion_matrix(y_test.astype(int), preds.astype(int), labels=list(range(len(LABEL_NAMES))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("LightGBM Tuned - Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision / recall / F1 from the metrics JSON
test_report = tuned_metrics["test"]["classification_report"]
rows = []
for idx, label in enumerate(LABEL_NAMES):
    cls = test_report.get(str(idx), {})
    rows.append({
        "workload_type": label,
        "precision": cls.get("precision", np.nan),
        "recall": cls.get("recall", np.nan),
        "f1": cls.get("f1-score", np.nan),
        "support": cls.get("support", np.nan),
    })
report_df = pd.DataFrame(rows)
display(report_df)

fi = pd.read_csv(PROJECT_ROOT / "models" / "classifier" / "lightgbm_tuned_feature_importance.csv")
top20 = fi.head(20).sort_values("importance")
plt.figure(figsize=(10, 8))
plt.barh(top20["feature"], top20["importance"], color="steelblue")
plt.title("Top 20 Feature Importances - Tuned LightGBM")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
# ARF metrics and prequential accuracy curves
with open(PROJECT_ROOT / "models" / "classifier" / "arf_metrics.json", "r", encoding="utf-8") as f:
    arf_metrics = json.load(f)
arf_curve = pd.read_csv(PROJECT_ROOT / "models" / "classifier" / "arf_prequential_accuracy.csv")

for split in ["train", "val", "test"]:
    print(f"ARF {split} prequential accuracy: {arf_metrics[split]['final_accuracy']:.4f}")

curve_plot = arf_curve[arf_curve["split"].isin(["train", "val"])]
plt.figure(figsize=(11, 6))
for split in ["train", "val"]:
    subset = curve_plot[curve_plot["split"] == split]
    plt.plot(subset["n_seen"], subset["accuracy"], marker="o", linewidth=1.5, label=split)
plt.title("ARF+ADWIN Prequential Accuracy Curve")
plt.xlabel("n_seen")
plt.ylabel("accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Side-by-side comparison table
comparison_rows = []
for idx, label in enumerate(LABEL_NAMES):
    lgb_f1 = test_report.get(str(idx), {}).get("f1-score", np.nan)
    arf_f1 = arf_metrics["test"]["per_class"].get(label, {}).get("f1", np.nan)
    comparison_rows.append({
        "workload_type": label,
        "LightGBM Tuned F1": lgb_f1,
        "ARF+ADWIN F1": arf_f1,
    })
comparison_df = pd.DataFrame(comparison_rows)
summary_df = pd.DataFrame([
    {"model": "LightGBM Tuned", "test_accuracy": test_acc},
    {"model": "ARF+ADWIN", "test_accuracy": arf_metrics["test"]["final_accuracy"]},
])
display(summary_df)
display(comparison_df)